In [1]:
%load_ext autoreload
%autoreload 2

from paper_utils import *
    
import os
os.chdir('../..')
from sklearn.metrics import cohen_kappa_score


In [2]:
runs = {
    'uvfbxm6d': ['squad-llama2', ['config.yaml', 'validation_generations.pkl']],
    'm3y3x605': ['squad-gpt35', ['config.yaml', 'uncertainty_measures.pkl']],
    '6cxqnx2u': ['squad-gpt4', ['config.yaml', 'uncertainty_measures.pkl']],
    'dcnndvny': ['squad-f1', ['config.yaml', 'uncertainty_measures.pkl']],

    
    'dqtye228': ['bioasq-llama2', ['config.yaml', 'validation_generations.pkl']],
    'nu83s9tf': ['bioasq-gpt35', ['config.yaml', 'uncertainty_measures.pkl']],
    'nybkuatj': ['bioasq-gpt4', ['config.yaml', 'uncertainty_measures.pkl']],
    '65uy3o6j': ['bioasq-f1', ['config.yaml', 'uncertainty_measures.pkl']],


    '1gxo9oef': ['trivia_qa-llama2', ['config.yaml', 'validation_generations.pkl']],
    'lgurwdvq': ['trivia_qa-gpt35', ['config.yaml', 'uncertainty_measures.pkl']],
    'zk37wfnl': ['trivia_qa-gpt4', ['config.yaml', 'uncertainty_measures.pkl']],
    '39mzv1st': ['trivia_qa-f1', ['config.yaml', 'uncertainty_measures.pkl']],
    
    # gpt-4 and f1 for trivia/bio are curently running
    # 171852--171855
    
}

all_configs, all_results = {}, {}
for wandb_id, (name, files) in runs.items():
    all_configs[wandb_id], all_results[wandb_id] = restore_file(wandb_id, filenames=files)

In [3]:
# print for human rating

# Choose dataset:
active_run = 'uvfbxm6d'
# active_run = 'dqtye228'
# active_run = '1gxo9oef'


N_ANNOTATIONS = 100
START_i = 0
END_i = 100

columns = ['index', 'data_id', 'wandb_id', 'dataset', 'truth_metric', 'truth_value']

for i in range(START_i, END_i):

    key = list(all_results[active_run].keys())[i]
    
    result = all_results[active_run][key]
    print(80 * f'-')
    print(f'{i} / {key} -------- New Question ------')
    print('Q:', result['question'])
    print('True answer:', result['reference']['answers']['text'])
    print('Model response:', result['most_likely_answer']['response'])


--------------------------------------------------------------------------------
0 / 57338007d058e614000b5bdb -------- New Question ------
Q: What was Warsaw's population in 1901?
True answer: ['711,988', '711,988', '711,988']
Model response: According to the 1901 census, Warsaw's population was approximately 750,000 people.
--------------------------------------------------------------------------------
1 / 571cc5c45efbb31900334dde -------- New Question ------
Q: When did O2 begin to acculturate in the atmosphere?
True answer: ['2.5 billion years ago', '2.5 billion years ago', 'about 2.5 billion years ago', 'about 2.5 billion years ago', '2.5 billion years ago during the Great Oxygenation Event']
Model response: O2 began to accumulate in the atmosphere around 2.7 billion years ago during the Great Oxygenation Event.
--------------------------------------------------------------------------------
2 / 5733a5f54776f41900660f46 -------- New Question ------
Q: Who was Frédéric Chopin?
True

In [7]:
gen_runs = {
    'squad': 'uvfbxm6d',
    'bioasq': 'dqtye228',
    'trivia_qa': '1gxo9oef',    
}


N_ANNOTATIONS = 100
START_i = 0
END_i = 100

# iterate over all to generate csv
users = []  # [human_jansen, human_sebhar]
runs_data = []

for active_run in runs:
    # if runs[active_run][0] not in ['trivia_qa-gpt4', 'bioasq-gpt4']:
    #     continue
    dataset, metric = runs[active_run][0].split('-')
    main_run = gen_runs[dataset]

    # only main run has ID
    print(active_run, main_run)

    for i in range(START_i, END_i):

        # get unique ID from main run
        key = list(all_results[main_run].keys())[i]        

        if runs[active_run][1][1] == 'uncertainty_measures.pkl':
            accuracy = 1 - all_results[active_run]['validation_is_false'][i]
        elif runs[active_run][1][1] == 'validation_generations.pkl':
            result = all_results[active_run][key]
            accuracy = result['most_likely_answer']['accuracy']
        else:
            raise

        accuracy = int(accuracy)
        run_data = [i, key, active_run, dataset, metric, accuracy]
        # if metric in WRITE_AUTOMATED_METRICS:
        runs_data.append(run_data)

        # print(','.join([str(i) for i in run_data]))
    
        for user in users:
            runs_data.append([i, key, run, dataset, f'human_{user}', 'FILL_IN'])

uvfbxm6d uvfbxm6d
m3y3x605 uvfbxm6d
6cxqnx2u uvfbxm6d
dcnndvny uvfbxm6d
dqtye228 dqtye228
nu83s9tf dqtye228
nybkuatj dqtye228
65uy3o6j dqtye228
1gxo9oef 1gxo9oef
lgurwdvq 1gxo9oef
zk37wfnl 1gxo9oef
39mzv1st 1gxo9oef


In [8]:
# [DONE] Use this to set up the CSV that you fill values in
print(pd.DataFrame(runs_data, columns=columns).sort_values(['dataset', 'truth_metric', 'index']).to_csv(index=False))

index,data_id,wandb_id,dataset,truth_metric,truth_value
0,6278cb6056bf9aee6f00000d,65uy3o6j,bioasq,f1,0
1,5177def18ed59a060a000034,65uy3o6j,bioasq,f1,0
2,5160412d298dcd4e5100003c,65uy3o6j,bioasq,f1,0
3,533581f5d6d3ac6a3400004d,65uy3o6j,bioasq,f1,0
4,5a7237672dc08e987e000008,65uy3o6j,bioasq,f1,0
5,5a7615af83b0d9ea6600001f,65uy3o6j,bioasq,f1,0
6,56be06cdef6e394741000004,65uy3o6j,bioasq,f1,0
7,601d73261cb411341a00003a,65uy3o6j,bioasq,f1,0
8,58cbb55402b8c60953000033,65uy3o6j,bioasq,f1,0
9,5c9e6e99ecadf2e73f000036,65uy3o6j,bioasq,f1,0
10,58dd07488acda34529000025,65uy3o6j,bioasq,f1,0
11,5178d6be8ed59a060a000038,65uy3o6j,bioasq,f1,0
12,553c011af321868558000009,65uy3o6j,bioasq,f1,0
13,5c929094ecadf2e73f000019,65uy3o6j,bioasq,f1,0
14,56e19eba51531f7e33000012,65uy3o6j,bioasq,f1,0
15,5c72a9147c78d6947100006e,65uy3o6j,bioasq,f1,0
16,5179602c8ed59a060a00003d,65uy3o6j,bioasq,f1,0
17,604915581cb411341a00016a,65uy3o6j,bioasq,f1,0
18,515993f3d24251bc050000a0,65uy3o6j,bioasq,f1,0
19,601eb3b61cb411341a00

# Load CSV and evaluate agreement

* bad labels in trivia_qa, maybe ignore
    * 11,sfq_20548--134/134_2666628.txt#0_0
    * 15,sfq_20731--116/116_1032584.txt#0_2,1gxo9oef,trivia_qa,human_jansen,0
    * 37 / odql_3253--63/63_150960.txt#0_2
    * 54 / sfq_8522--31/31_699713.txt#0_1
    * 70 / qb_3569--170/170_366697.txt#0_2


In [9]:
df = pd.read_csv('notebooks/paper_evals/23-11-24-accuracy-evaluation.csv', index_col=None)
# kick out bad trivia_qa labels
df = df[~ ((df.dataset == 'trivia_qa') & df['index'].map(lambda x: x in [11, 15, 37, 54]))]

In [10]:
# df = pd.read_csv('notebooks/paper_evals/23-11-24-accuracy-evaluation.csv', index_col=None)
df = pd.read_csv('notebooks/paper_evals/23-11-24-accuracy-evaluation.csv', index_col=None)

def remove_incomplete(df):
    # filter out incomplete data!
    ignore = []
    for metric, mdf in df.groupby('truth_metric'):
        if (mdf.truth_value == 'FILL_IN').any():
            ignore.append(metric)
    print(f'Ignoring metrics {ignore} for now.')
    df = df[df.truth_metric.map(lambda x: x not in ignore)]
    return df

df = remove_incomplete(df)

# check for completeness
for name, tmp in df.groupby(['index', 'dataset', 'truth_metric']):
    if len(tmp) > 1:
        print(name)
        display(tmp)

if df.truth_value.nunique() > 2:
    raise ValueError

df

Ignoring metrics [] for now.


,index,data_id,wandb_id,dataset,truth_metric,truth_value
0,0,57338007d058e614000b5bdb,m3y3x605,squad,human_jansen,0
1,1,571cc5c45efbb31900334dde,m3y3x605,squad,human_jansen,0
2,2,5733a5f54776f41900660f46,m3y3x605,squad,human_jansen,1
3,3,5727dd2e4b864d1900163eba,m3y3x605,squad,human_jansen,0
4,4,5733fd66d058e614000b6737,m3y3x605,squad,human_jansen,0
...,...,...,...,...,...,...
1895,95,qg_51--144/144_2513486.txt#0_2,1gxo9oef,trivia_qa,llama2,1
1896,96,odql_4978--60/60_1891835.txt#0_2,1gxo9oef,trivia_qa,llama2,1
1897,97,qz_5346--59/59_2610312.txt#0_0,1gxo9oef,trivia_qa,llama2,1
1898,98,qw_2135--103/103_649320.txt#0_2,1gxo9oef,trivia_qa,llama2,1


In [13]:
def agreement(x, y):
    return np.mean(x == y)

def style(df):
    return df.style.background_gradient(sns.color_palette('crest', as_cmap=True), axis=None).format(precision=2)


for method in [cohen_kappa_score, agreement, 'pearson', 'kendall', 'spearman']:
    print(90*'x')
    print(colorize(f'METHOD: {method}'))
    print(90*'x')

    corrs = []
    for dataset, gdf in df.groupby('dataset'):
        pdf = gdf.pivot(index='index', columns='truth_metric', values='truth_value')
        corr = pdf.corr(method=method)
        corrs.append(corr)
        print(colorize(f'method: {method} -- dataset: {dataset}', 1))
        display(style(corr))

    avg = pd.concat(corrs).reset_index().groupby('truth_metric').mean()
    # print(colorize(f'METHOD: {method} --average_over_datasets', 2))
    # display(style(avg))

    tmp = pd.DataFrame([0.5 * (avg.loc['human_jansen'] + avg.loc['human_sebhar'])])
    tmp.index = ['human_average']
    avg = avg._append(tmp)
    print(colorize(f'METHOD: {method} --average_over_datasets', 2))
    display(style(avg))
    

xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: <function cohen_kappa_score at 0x7f331c685940>
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
method: <function cohen_kappa_score at 0x7f331c685940> -- dataset: bioasq


truth_metric,f1,gpt35,gpt4,human_jansen,human_sebhar,llama2
truth_metric,,,,,,
f1,1.00,0.01,0.01,0.01,0.01,0.02
gpt35,0.01,1.00,0.89,0.71,0.87,0.81
gpt4,0.01,0.89,1.00,0.73,0.89,0.79
human_jansen,0.01,0.71,0.73,1.00,0.79,0.78
human_sebhar,0.01,0.87,0.89,0.79,1.00,0.90
llama2,0.02,0.81,0.79,0.78,0.90,1.00


method: <function cohen_kappa_score at 0x7f331c685940> -- dataset: squad


truth_metric,f1,gpt35,gpt4,human_jansen,human_kunda,human_sebhar,llama2
truth_metric,,,,,,,
f1,1.00,0.04,0.06,0.06,0.11,0.07,0.08
gpt35,0.04,1.00,0.70,0.76,0.61,0.69,0.69
gpt4,0.06,0.70,1.00,0.84,0.64,0.72,0.72
human_jansen,0.06,0.76,0.84,1.00,0.75,0.79,0.83
human_kunda,0.11,0.61,0.64,0.75,1.00,0.77,0.58
human_sebhar,0.07,0.69,0.72,0.79,0.77,1.00,0.76
llama2,0.08,0.69,0.72,0.83,0.58,0.76,1.00


method: <function cohen_kappa_score at 0x7f331c685940> -- dataset: trivia_qa


...

truth_metric,f1,gpt35,gpt4,human_jansen,human_kunda,human_sebhar,llama2
truth_metric,,,,,,,
f1,1.00,0.10,0.13,0.13,0.24,0.14,0.16
gpt35,0.10,1.00,0.70,0.77,0.62,0.71,0.71
gpt4,0.13,0.70,1.00,0.84,0.64,0.73,0.72
human_jansen,0.13,0.77,0.84,1.00,0.76,0.79,0.84
human_kunda,0.24,0.62,0.64,0.76,1.00,0.78,0.59
human_sebhar,0.14,0.71,0.73,0.79,0.78,1.00,0.76
llama2,0.16,0.71,0.72,0.84,0.59,0.76,1.00


method: spearman -- dataset: trivia_qa


truth_metric,f1,gpt35,gpt4,human_jansen,human_sebhar,llama2
truth_metric,,,,,,
f1,1.00,0.12,0.17,0.19,0.17,0.16
gpt35,0.12,1.00,0.69,0.65,0.60,0.68
gpt4,0.17,0.69,1.00,0.87,0.86,0.81
human_jansen,0.19,0.65,0.87,1.00,0.80,0.76
human_sebhar,0.17,0.60,0.86,0.80,1.00,0.81
llama2,0.16,0.68,0.81,0.76,0.81,1.00


METHOD: spearman --average_over_datasets


truth_metric,f1,gpt35,gpt4,human_jansen,human_sebhar,llama2,human_kunda
f1,1.00,0.10,0.13,0.13,0.13,0.13,0.24
gpt35,0.10,1.00,0.76,0.71,0.73,0.74,0.62
gpt4,0.13,0.76,1.00,0.81,0.83,0.78,0.64
human_jansen,0.13,0.71,0.81,1.00,0.79,0.79,0.76
human_kunda,0.24,0.62,0.64,0.76,0.78,0.59,1.00
human_sebhar,0.13,0.73,0.83,0.79,1.00,0.83,0.78
llama2,0.13,0.74,0.78,0.79,0.83,1.00,0.59
human_average,0.13,0.72,0.82,0.90,0.90,0.81,0.77


# Notes

* On Cohen's Cappa from Wikipedia:

> Nonetheless, magnitude guidelines have appeared in the literature. Perhaps the first was Landis and Koch,[16] who characterized values < 0 as indicating no agreement and 0–0.20 as slight, 0.21–0.40 as fair, 0.41–0.60 as moderate, 0.61–0.80 as substantial, and 0.81–1 as almost perfect agreement.

* Squad-f1 completely fails for long generations!

 
## Some thoughts on evaluation:

We are checking  if the answer matches the expected answer, not if the answer is correct!

bad_questions: 

note, that I'm not including underspecified questions 
note, that for these questions, we still just check if the model matches the 'true answer' as given by the data (even if that is wrong)

```
index, data_id, reason, explanation
11, 57300a9a04bcaa1900d77065, 'True Answer Incorrect', "Quote from Wikipedia: 'The Versailles Treaty also stipulated that Allied military forces would withdraw from the Rhineland by 1935.' And in 1936 Germany re-militarized the Rhineland"
```